# Lamoda × ЦУ — этап 1: обзор данных и предобработка

Ноутбук ничего не обучает. Он отвечает на вопросы:

1. Что лежит в файлах, есть ли пропуски, дубли и странные значения.
2. Какой период покрывают заказы и есть ли сезонность.
3. Пересекаются ли клиенты между срезами train и test.
4. Какие заказы лежат в файле относительно даты скоринга каждой строки (история, 14-дневный лаг, целевое окно).
5. Можно ли восстановить `target` из заказов.
6. Насколько сильны простейшие сигналы: давность и частота.

В конце сохраняются две предобработанные таблицы: `order_lines.parquet` и `order_tbl.parquet`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("LAMODA_OUT_DIR", "."))

GAP_DAYS = 14          # лаг между последним допустимым заказом и scoring_date
HISTORY_YEARS = 2      # глубина допустимой истории
TARGET_MONTHS = 3      # длина целевого окна

## 1. Загрузка и общий вид таблиц

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

tables = {"train": train, "test": test, "sample_submission": sample, "orders": orders, "products": products}


def overview(df):
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing": df.isna().sum(),
            "missing_%": df.isna().mean() * 100,
            "nunique": df.nunique(),
        }
    )


for name, df in tables.items():
    mem = df.memory_usage(deep=True).sum() / 2**20
    print(f"\n=== {name}: {df.shape[0]:,} строк x {df.shape[1]} колонок, {mem:.1f} МБ")
    display(overview(df))
    display(df.head(3))

## 2. Ключи, дубли и связи между таблицами

In [ ]:
checks = {}

checks["train: дубли (customer_id, scoring_date)"] = int(train.duplicated(["customer_id", "scoring_date"]).sum())
checks["test: дубли (customer_id, scoring_date)"] = int(test.duplicated(["customer_id", "scoring_date"]).sum())
checks["test: дубли customer_id"] = int(test["customer_id"].duplicated().sum())
checks["sample: набор ID совпадает с test"] = set(sample["customer_id"]) == set(test["customer_id"])
checks["sample: порядок ID совпадает с test"] = bool(
    len(sample) == len(test) and (sample["customer_id"].to_numpy() == test["customer_id"].to_numpy()).all()
)

checks["products: дубли product_id"] = int(products["product_id"].duplicated().sum())
checks["orders: полные дубли строк"] = int(orders.duplicated().sum())
checks["orders: дубли (order_id, product_id)"] = int(orders.duplicated(["order_id", "product_id"]).sum())

per_order = orders.groupby("order_id").agg(
    n_customers=("customer_id", "nunique"),
    n_dates=("order_date", "nunique"),
    n_lines=("product_id", "size"),
)
checks["orders: заказов с >1 клиентом"] = int((per_order["n_customers"] > 1).sum())
checks["orders: заказов с >1 датой"] = int((per_order["n_dates"] > 1).sum())

known_products = set(products["product_id"])
checks["orders: доля позиций без товара в справочнике, %"] = round(
    float((~orders["product_id"].isin(known_products)).mean() * 100), 4
)
checks["products: товаров, которых нет в заказах"] = int((~products["product_id"].isin(orders["product_id"])).sum())

slice_customers = set(train["customer_id"]) | set(test["customer_id"])
order_customers = set(orders["customer_id"])
checks["клиентов в orders всего"] = len(order_customers)
checks["клиентов в orders, которых нет в train/test"] = len(order_customers - slice_customers)
checks["клиентов train/test без единой строки в orders"] = len(slice_customers - order_customers)

display(pd.Series(checks, name="значение").to_frame())

print("Позиций в заказе:")
display(per_order["n_lines"].describe(percentiles=[0.5, 0.9, 0.99]).to_frame().T)

## 3. Значения колонок

In [ ]:
num_cols = ["paid_price", "loyalty_percent", "discount_percent"]
display(orders[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

display(
    pd.DataFrame(
        {
            "< 0": (orders[num_cols] < 0).sum(),
            "== 0": (orders[num_cols] == 0).sum(),
            "== 0, %": (orders[num_cols] == 0).mean() * 100,
            "> 100": (orders[num_cols] > 100).sum(),
        }
    )
)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].hist(np.log1p(orders["paid_price"].clip(lower=0)), bins=80)
axes[0].set_title("log1p(paid_price)")
axes[1].hist(orders["loyalty_percent"].dropna(), bins=50)
axes[1].set_title("loyalty_percent")
axes[2].hist(orders["discount_percent"].dropna(), bins=50)
axes[2].set_title("discount_percent")
plt.tight_layout()
plt.show()

In [ ]:
for col in [c for c in products.columns if c != "product_id"]:
    vc = products[col].value_counts(dropna=False)
    print(f"\n--- products.{col}: {products[col].nunique()} уникальных, пропусков {products[col].isna().sum()}")
    display(vc.head(12).to_frame("товаров").T)

## 4. Период данных и сезонность

In [ ]:
print("order_date:", orders["order_date"].min().date(), "…", orders["order_date"].max().date())
print("scoring_date в train:", sorted(train["scoring_date"].dt.date.unique()))
print("scoring_date в test: ", sorted(test["scoring_date"].dt.date.unique()))
print("Доля позиций с ненулевым временем в order_date, %:",
      round(float((orders["order_date"] != orders["order_date"].dt.normalize()).mean() * 100), 4))

monthly = (
    orders.assign(month=orders["order_date"].dt.to_period("M"))
    .groupby("month")
    .agg(
        lines=("product_id", "size"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
        paid_sum=("paid_price", "sum"),
    )
)
monthly["avg_order_value"] = monthly["paid_sum"] / monthly["orders"]
display(monthly)

ax = monthly[["orders", "customers"]].plot(figsize=(14, 4), marker="o", title="Заказы и активные клиенты по месяцам")
for d in sorted(set(train["scoring_date"]) | set(test["scoring_date"])):
    ax.axvline(pd.Timestamp(d).to_period("M").ordinal, color="gray", ls="--", lw=0.8)
plt.show()

daily = orders.groupby("order_date")["order_id"].nunique()
daily.loc[daily.index >= daily.index.max() - pd.Timedelta(days=150)].plot(
    figsize=(14, 3), title="Заказы по дням: последние 150 дней данных"
)
plt.show()

dow = orders.drop_duplicates("order_id")["order_date"].dt.dayofweek.value_counts(normalize=True).sort_index()
print("Доля заказов по дням недели (0 = понедельник):", dow.round(3).to_dict())

## 5. Срезы и пересечение клиентов

Если один клиент встречается в нескольких срезах, его целевые окна перекрываются. Это важно и для валидации (нужна группировка по клиенту), и как источник утечки.

In [ ]:
slices = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
slices["slice"] = slices["scoring_date"].dt.strftime("%Y-%m-%d")

display(
    slices.groupby(["part", "slice"]).agg(
        rows=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        target_rate=("target", "mean"),
    )
)

ids = {s: set(g["customer_id"]) for s, g in slices.groupby("slice")}
names = sorted(ids)
inter = pd.DataFrame([[len(ids[a] & ids[b]) for b in names] for a in names], index=names, columns=names)
print("Пересечение клиентов между срезами (на диагонали — размер среза):")
display(inter)

n_slices = slices.groupby("customer_id")["slice"].nunique()
print("Сколько клиентов встречается в 1, 2, 3, 4 срезах:", n_slices.value_counts().sort_index().to_dict())
print("Доля клиентов test, которые есть в train, %:",
      round(float(test["customer_id"].isin(train["customer_id"]).mean() * 100), 2))

In [ ]:
# Насколько согласованы метки одного клиента в соседних срезах train
tw = train.pivot_table(index="customer_id", columns=train["scoring_date"].dt.strftime("%Y-%m-%d"), values="target")
cols = list(tw.columns)
for a, b in zip(cols[:-1], cols[1:]):
    both = tw[[a, b]].dropna()
    if len(both) == 0:
        print(f"{a} и {b}: общих клиентов нет")
        continue
    print(f"\n{a} -> {b}: общих клиентов {len(both):,}, метки совпадают у {(both[a] == both[b]).mean():.1%}")
    display(pd.crosstab(both[a], both[b], normalize="index"))

## 6. Таблица заказов и их положение относительно даты скоринга

Сворачиваем позиции до уровня заказа, затем для каждой строки `(customer_id, scoring_date)` раскладываем заказы клиента по зонам:

| Зона | Интервал | Можно использовать в признаках |
| --- | --- | --- |
| `0_too_old` | раньше `scoring_date − 2 года − 14 дней` | нет |
| `1_history` | до `scoring_date − 14 дней` включительно | да |
| `2_gap` | последние 14 дней перед `scoring_date` | нет |
| `3_target` | `(scoring_date, scoring_date + 3 месяца]` | нет, это и есть таргет |
| `4_after` | позже целевого окна | нет |

In [ ]:
order_tbl = (
    orders.groupby(["customer_id", "order_id"], as_index=False)
    .agg(
        order_date=("order_date", "min"),
        order_sum=("paid_price", "sum"),
        n_items=("product_id", "size"),
        n_products=("product_id", "nunique"),
        discount_mean=("discount_percent", "mean"),
        discount_max=("discount_percent", "max"),
        loyalty_mean=("loyalty_percent", "mean"),
    )
    .sort_values(["customer_id", "order_date", "order_id"])
    .reset_index(drop=True)
)
print(f"Заказов: {len(order_tbl):,}, клиентов: {order_tbl['customer_id'].nunique():,}")
display(order_tbl[["order_sum", "n_items", "discount_mean", "loyalty_mean"]].describe(percentiles=[0.05, 0.5, 0.95, 0.99]).T)

per_customer = order_tbl.groupby("customer_id").agg(n_orders=("order_id", "size"), n_days=("order_date", "nunique"))
print("Заказов на клиента за всё время:")
display(per_customer["n_orders"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).to_frame().T)
print("Доля клиентов, у которых бывает несколько заказов в один день, %:",
      round(float((per_customer["n_orders"] > per_customer["n_days"]).mean() * 100), 2))

In [ ]:
bounds = pd.DataFrame({"scoring_date": sorted(slices["scoring_date"].unique())})
bounds["cutoff"] = bounds["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
bounds["history_start"] = bounds["scoring_date"].map(lambda d: d - pd.DateOffset(years=HISTORY_YEARS)) - pd.Timedelta(days=GAP_DAYS)
bounds["target_end"] = bounds["scoring_date"].map(lambda d: d + pd.DateOffset(months=TARGET_MONTHS))
display(bounds)

pairs = (
    slices.merge(bounds, on="scoring_date", how="left")
    .merge(order_tbl[["customer_id", "order_id", "order_date"]], on="customer_id", how="left")
)
od = pairs["order_date"]
pairs["zone"] = np.select(
    [
        od.isna(),
        od < pairs["history_start"],
        od <= pairs["cutoff"],
        od <= pairs["scoring_date"],
        od <= pairs["target_end"],
    ],
    ["no_orders", "0_too_old", "1_history", "2_gap", "3_target"],
    default="4_after",
)

zone_orders = pairs.pivot_table(index="slice", columns="zone", values="order_id", aggfunc="count", fill_value=0)
print("Число заказов по зонам:")
display(zone_orders)

zone_customers = (
    pairs.drop_duplicates(["slice", "customer_id", "zone"])
    .pivot_table(index="slice", columns="zone", values="customer_id", aggfunc="count", fill_value=0)
)
zone_customers = zone_customers.div(slices.groupby("slice").size(), axis=0) * 100
print("Доля клиентов среза, у которых есть хотя бы один заказ в зоне, %:")
display(zone_customers)

print("Последняя дата заказа у клиентов каждого среза:")
display(pairs.groupby("slice")["order_date"].max().to_frame("max_order_date"))

### Восстанавливается ли `target` из заказов

Если метка совпадает с правилом «есть заказ в `(scoring_date, scoring_date + 3 месяца]`», то мы точно понимаем её определение и можем сами строить дополнительные обучающие срезы на более ранних датах.

In [ ]:
recon = (
    pairs.assign(in_target=pairs["zone"].eq("3_target"))
    .groupby(["slice", "customer_id"], as_index=False)["in_target"]
    .any()
    .rename(columns={"in_target": "target_from_orders"})
)
chk = train.assign(slice=train["scoring_date"].dt.strftime("%Y-%m-%d")).merge(recon, on=["slice", "customer_id"], how="left")
chk["target_from_orders"] = chk["target_from_orders"].fillna(False).astype(int)

print(f"Совпадение target с восстановленным по заказам: {(chk['target'] == chk['target_from_orders']).mean():.4%}")
display(pd.crosstab([chk["slice"], chk["target"]], chk["target_from_orders"]))

test_recon = recon.merge(test.assign(slice=test["scoring_date"].dt.strftime("%Y-%m-%d"))[["slice", "customer_id"]])
print(f"Клиентов test с заказами в целевом окне по файлу orders: {test_recon['target_from_orders'].mean():.4%} "
      "(ожидаем 0%, иначе в данных утечка)")

## 7. Первые сигналы: давность и частота

Считаем только по зоне `1_history`. AP@20% одиночных признаков даёт нижнюю планку для будущих моделей.

In [ ]:
def average_precision_at_20_percent(customer_id, target, score):
    ranking = (
        pd.DataFrame({"customer_id": np.asarray(customer_id).astype(str), "target": np.asarray(target), "score": np.asarray(score)})
        .sort_values(["score", "customer_id"], ascending=[False, True], kind="mergesort")
        .reset_index(drop=True)
    )
    k = int(np.ceil(len(ranking) * 0.20))
    top_target = ranking["target"].iloc[:k].to_numpy()
    precision_at_rank = np.cumsum(top_target) / np.arange(1, k + 1)
    denominator = min(k, int(ranking["target"].sum()))
    return (precision_at_rank * top_target).sum() / denominator


hist = pairs[pairs["zone"].eq("1_history")].copy()
hist["days_before_cutoff"] = (hist["cutoff"] - hist["order_date"]).dt.days
rf = hist.groupby(["slice", "customer_id"]).agg(
    recency=("days_before_cutoff", "min"),
    tenure=("days_before_cutoff", "max"),
    n_orders=("order_id", "nunique"),
    n_orders_90d=("days_before_cutoff", lambda s: int((s <= 90).sum())),
)
quick = slices.merge(rf.reset_index(), on=["slice", "customer_id"], how="left")
quick["has_history"] = quick["recency"].notna()

print("Доля строк без единого заказа в допустимой истории, %:")
display((1 - quick.groupby(["part", "slice"])["has_history"].mean()).mul(100).to_frame("no_history_%"))
display(quick.groupby(["part", "slice"])[["recency", "tenure", "n_orders", "n_orders_90d"]].describe(percentiles=[0.5]).T)

In [ ]:
qt = quick[quick["part"].eq("train")].copy()
REC_EDGES = [-1, 7, 14, 30, 60, 90, 180, 365, 10_000]
REC_LABELS = ["0-7", "8-14", "15-30", "31-60", "61-90", "91-180", "181-365", "365+"]
qt["recency_bin"] = pd.cut(qt["recency"], REC_EDGES, labels=REC_LABELS).astype(str)
qt.loc[~qt["has_history"], "recency_bin"] = "нет истории"
qt["n_orders_bin"] = qt["n_orders"].fillna(0).clip(upper=10).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
order_bins = REC_LABELS + ["нет истории"]
by_rec = qt.groupby("recency_bin")["target"].agg(["mean", "size"]).reindex(order_bins).dropna()
by_rec["mean"].plot.bar(ax=axes[0], title="Доля target = 1 по давности последнего заказа, дни")
by_freq = qt.groupby("n_orders_bin")["target"].agg(["mean", "size"])
by_freq["mean"].plot.bar(ax=axes[1], title="Доля target = 1 по числу заказов в истории (10 = 10+)")
plt.tight_layout()
plt.show()
display(by_rec.T)
display(by_freq.T)

rows = []
for s, g in qt.groupby("slice"):
    rec = g["recency"].fillna(10_000)
    rows.append(
        {
            "slice": s,
            "target_rate": g["target"].mean(),
            "случайно": average_precision_at_20_percent(g["customer_id"], g["target"], np.random.default_rng(0).random(len(g))),
            "-recency": average_precision_at_20_percent(g["customer_id"], g["target"], -rec),
            "n_orders": average_precision_at_20_percent(g["customer_id"], g["target"], g["n_orders"].fillna(0)),
            "n_orders_90d": average_precision_at_20_percent(g["customer_id"], g["target"], g["n_orders_90d"].fillna(0)),
            "n_orders / (recency + 30)": average_precision_at_20_percent(
                g["customer_id"], g["target"], g["n_orders"].fillna(0) / (rec + 30)
            ),
        }
    )
print("AP@20% одиночных признаков по срезам train:")
display(pd.DataFrame(rows).set_index("slice"))

## 8. Предобработка и сохранение

- `order_lines.parquet` — позиции заказов с атрибутами товара, компактные типы, пропуски в атрибутах заменены на `"unknown"`.
- `order_tbl.parquet` — одна строка на заказ.

Фильтрация по дате отсечения здесь не делается: она своя для каждой строки `(customer_id, scoring_date)` и выполняется на этапе признаков.

In [ ]:
attr_cols = [c for c in products.columns if c != "product_id"]
products_clean = products.drop_duplicates("product_id").copy()
for c in attr_cols:
    products_clean[c] = products_clean[c].astype("object").where(products_clean[c].notna(), "unknown").astype(str)

order_lines = orders.merge(products_clean, on="product_id", how="left", validate="m:1")
assert len(order_lines) == len(orders), "merge изменил число строк"
for c in attr_cols:
    order_lines[c] = order_lines[c].fillna("unknown").astype("category")
for c in ["paid_price", "loyalty_percent", "discount_percent"]:
    order_lines[c] = order_lines[c].astype("float32")
order_lines = order_lines.sort_values(["customer_id", "order_date", "order_id"]).reset_index(drop=True)

order_lines.to_parquet(OUT_DIR / "order_lines.parquet", index=False)
order_tbl.to_parquet(OUT_DIR / "order_tbl.parquet", index=False)

for name, df in {"order_lines": order_lines, "order_tbl": order_tbl}.items():
    print(f"{name}: {df.shape}, {df.memory_usage(deep=True).sum() / 2**20:.1f} МБ")
display(order_lines.head())
display(order_tbl.head())